In [1]:
# I imported the required libraries.
import tkinter as tk
from tkinter import ttk, messagebox
from tkinter.scrolledtext import ScrolledText
import pandas as pd
import numpy as np
import joblib
import csv
from datetime import datetime



# I loaded the trained CatBoost model.
model = joblib.load("Early_Mental_Health_Prediction_Model.pkl")

# I loaded the saved label encoders.
label_encoders = joblib.load("Early_Mental_Health_Label_Encoders.pkl")



# I created the main application window.
root = tk.Tk()

# I set the application title.
root.title("Early Mental Health Prediction Prototype")

# I set the application size.
# I set the initial application size.
root.geometry("1400x850")

# I allowed the window to be resized.
root.resizable(True, True)

# I enabled the maximized window.
try:
    root.state("zoomed")          # Windows
except:
    root.attributes("-zoomed", True)   # Linux

# I changed the background colour.
root.configure(bg="white")



TITLE_FONT = ("Segoe UI", 22, "bold")
HEADER_FONT = ("Segoe UI", 13, "bold")
LABEL_FONT = ("Segoe UI", 10)
BUTTON_FONT = ("Segoe UI", 10, "bold")
RESULT_FONT = ("Segoe UI", 12, "bold")



title = tk.Label(
    root,
    text="Early Mental Health Prediction Using Machine Learning",
    font=TITLE_FONT,
    bg="white",
    fg="#1F4E79"
)

title.pack(pady=8)



main_frame = tk.Frame(
    root,
    bg="white"
)

main_frame.pack(fill="both", expand=True)


left_frame = tk.LabelFrame(
    main_frame,
    text="Input Parameters",
    font=HEADER_FONT,
    bg="white",
    padx=20,
    pady=20
)

left_frame.pack(
    side="left",
    padx=20,
    pady=10,
    fill="both"
)



right_frame = tk.LabelFrame(
    main_frame,
    text="Prediction Result",
    font=HEADER_FONT,
    bg="white",
    padx=10,
    pady=10
)

right_frame.pack(
    side="right",
    padx=10,
    pady=10,
    fill="both",
    expand=True
)



def encode_value(column, value):

    if column in label_encoders:
        return label_encoders[column].transform([value])[0]

    return value


def predict():

    try:

        values = []

        for column in feature_columns:

            value = input_widgets[column].get()

            if value == "":

                messagebox.showwarning(
                    "Input Error",
                    f"Please select {column}."
                )

                return

            values.append(
                encode_value(column, value)
            )

        input_df = pd.DataFrame(
            [values],
            columns=feature_columns
        )

        prediction = model.predict(input_df)[0]

        probability = model.predict_proba(input_df)[0]

        confidence = np.max(probability) * 100

        if prediction == 1:

            result = "Treatment Required"

        else:

            result = "No Treatment Required"

        prediction_label.config(
            text=result,
            fg="#0B6E4F"
        )

        confidence_label.config(
            text=f"Confidence : {confidence:.2f}%"
        )

    except Exception as e:

        messagebox.showerror(
            "Prediction Error",
            str(e)
        )


# I created the feature list.
feature_columns = [
    "Gender",
    "Country",
    "Occupation",
    "self_employed",
    "family_history",
    "Days_Indoors",
    "Growing_Stress",
    "Changes_Habits",
    "Mental_Health_History",
    "Mood_Swings",
    "Coping_Struggles",
    "Work_Interest",
    "Social_Weakness",
    "mental_health_interview",
    "care_options"
]



# I created a dictionary to store all input widgets.
input_widgets = {}



# I created the input labels and drop-down boxes.
row = 0

for column in feature_columns:

    tk.Label(
        left_frame,
        text=column.replace("_", " "),
        font=LABEL_FONT,
        bg="white"
    ).grid(
        row=row,
        column=0,
        sticky="w",
        padx=10,
        pady=6
    )

    # I loaded the original dataset.
    original_df = pd.read_csv("Mental_Health_Dataset.csv")



    # I retrieved the original category values.
    values = sorted(
    original_df[column]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

    combo = ttk.Combobox(
        left_frame,
        values=values,
        width=35,
        state="readonly"
    )

    combo.grid(
        row=row,
        column=1,
        padx=10,
        pady=6
    )

    combo.current(0)

    input_widgets[column] = combo

    row += 1



# I created the prediction heading.
prediction_heading = tk.Label(
    right_frame,
    text="Prediction",
    font=("Segoe UI", 16, "bold"),
    bg="white",
    fg="#1F4E79"
)

prediction_heading.pack(pady=8)

# I created the prediction result label.
prediction_label = tk.Label(
    right_frame,
    text="Waiting for Prediction...",
    font=("Segoe UI", 18, "bold"),
    bg="white",
    fg="black"
)

prediction_label.pack(pady=5)

# I created the confidence label.
confidence_label = tk.Label(
    right_frame,
    text="Confidence : 0%",
    font=("Segoe UI", 14),
    bg="white"
)

confidence_label.pack(pady=4)



history_title = tk.Label(
    right_frame,
    text="Prediction History",
    font=("Segoe UI", 13, "bold"),
    bg="white",
    fg="#1F4E79"
)

history_title.pack(pady=(10, 5))

history_box = tk.Listbox(
    right_frame,
    width=60,
    height=6,
    font=("Consolas", 10)
)

history_box.pack(pady=10)



def clear_inputs():

    for widget in input_widgets.values():

        widget.current(0)

    prediction_label.config(
        text="Waiting for Prediction...",
        fg="black"
    )

    confidence_label.config(
        text="Confidence : 0%"
    )



def exit_application():

    root.destroy()



# I stored the original prediction function.
old_predict = predict

# I updated the prediction function.
def predict():

    old_predict()

    history_box.insert(
        tk.END,
        f"{datetime.now().strftime('%H:%M:%S')}  |  {prediction_label['text']}  |  {confidence_label['text']}"
    )



button_frame = tk.Frame(
    right_frame,
    bg="white"
)

button_frame.pack(pady=10)

predict_button = tk.Button(
    button_frame,
    text="Predict",
    command=predict,
    width=15,
    bg="#1F77B4",
    fg="white",
    font=BUTTON_FONT
)

predict_button.grid(row=0, column=0, padx=8)

clear_button = tk.Button(
    button_frame,
    text="Clear",
    command=clear_inputs,
    width=15,
    bg="#F39C12",
    fg="white",
    font=BUTTON_FONT
)

clear_button.grid(row=0, column=1, padx=8)

exit_button = tk.Button(
    button_frame,
    text="Exit",
    command=exit_application,
    width=15,
    bg="#C0392B",
    fg="white",
    font=BUTTON_FONT
)

exit_button.grid(row=0, column=2, padx=8)


# I created the human evaluation frame.
evaluation_frame = tk.LabelFrame(
    right_frame,
    text="Human Evaluation",
    font=HEADER_FONT,
    bg="white",
    padx=15,
    pady=15
)

evaluation_frame.pack(fill="x", padx=10, pady=10)



rating_options = [
    "1 - Poor",
    "2 - Fair",
    "3 - Good",
    "4 - Very Good",
    "5 - Excellent"
]



tk.Label(
    evaluation_frame,
    text="Ease of Use",
    font=LABEL_FONT,
    bg="white"
).grid(row=0, column=0, sticky="w", pady=5)

ease_combo = ttk.Combobox(
    evaluation_frame,
    values=rating_options,
    width=25,
    state="readonly"
)

ease_combo.grid(row=0, column=1, padx=10, pady=5)
ease_combo.current(2)


tk.Label(
    evaluation_frame,
    text="Interface Design",
    font=LABEL_FONT,
    bg="white"
).grid(row=1, column=0, sticky="w", pady=5)

design_combo = ttk.Combobox(
    evaluation_frame,
    values=rating_options,
    width=25,
    state="readonly"
)

design_combo.grid(row=1, column=1, padx=10, pady=5)
design_combo.current(2)



tk.Label(
    evaluation_frame,
    text="Prediction Satisfaction",
    font=LABEL_FONT,
    bg="white"
).grid(row=2, column=0, sticky="w", pady=3)

accuracy_combo = ttk.Combobox(
    evaluation_frame,
    values=rating_options,
    width=25,
    state="readonly"
)

accuracy_combo.grid(row=2, column=1, padx=10, pady=5)
accuracy_combo.current(2)



tk.Label(
    evaluation_frame,
    text="Overall Experience",
    font=LABEL_FONT,
    bg="white"
).grid(row=3, column=0, sticky="w", pady=5)

overall_combo = ttk.Combobox(
    evaluation_frame,
    values=rating_options,
    width=25,
    state="readonly"
)

overall_combo.grid(row=3, column=1, padx=10, pady=5)
overall_combo.current(2)


tk.Label(
    evaluation_frame,
    text="Comments",
    font=LABEL_FONT,
    bg="white"
).grid(row=4, column=0, sticky="nw", pady=10)

comments_box = ScrolledText(
    evaluation_frame,
    width=38,
    height=1,
    font=("Segoe UI", 10)
)

comments_box.grid(
    row=4,
    column=1,
    padx=5,
    pady=5
)



def save_feedback():

    try:

        # I collected the evaluation values.
        ease = ease_combo.get()
        design = design_combo.get()
        accuracy = accuracy_combo.get()
        overall = overall_combo.get()
        comments = comments_box.get("1.0", tk.END).strip()

        # I checked whether the CSV file already exists.
        import os

        file_exists = os.path.isfile("Human_Evaluation.csv")

        # I opened the CSV file.
        with open(
            "Human_Evaluation.csv",
            "a",
            newline="",
            encoding="utf-8"
        ) as file:

            writer = csv.writer(file)

            # I wrote the header if the file did not exist.
            if not file_exists:

                writer.writerow([
                    "Date",
                    "Time",
                    "Prediction",
                    "Confidence",
                    "Ease_of_Use",
                    "Interface_Design",
                    "Prediction_Satisfaction",
                    "Overall_Experience",
                    "Comments"
                ])

            # I wrote the evaluation results.
            writer.writerow([
                datetime.now().strftime("%d-%m-%Y"),
                datetime.now().strftime("%H:%M:%S"),
                prediction_label["text"],
                confidence_label["text"],
                ease,
                design,
                accuracy,
                overall,
                comments
            ])

        # I displayed a success message.
        messagebox.showinfo(
            "Success",
            "Human evaluation was submitted successfully."
        )

        # I cleared the comments.
        comments_box.delete("1.0", tk.END)

        # I reset the ratings.
        ease_combo.current(2)
        design_combo.current(2)
        accuracy_combo.current(2)
        overall_combo.current(2)

    except Exception as e:

        messagebox.showerror(
            "Error",
            str(e)
        )




submit_button = tk.Button(
    evaluation_frame,
    text="Submit Evaluation",
    command=save_feedback,
    bg="#27AE60",
    fg="white",
    font=BUTTON_FONT,
    width=22,
    height=2
)

submit_button.grid(
    row=5,
    column=0,
    columnspan=2,
    pady=5
)



footer = tk.Label(
    root,
    text="Machine Learning-Based Early Mental Health Prediction Prototype",
    font=("Segoe UI", 10),
    bg="white",
    fg="gray40"
)

footer.pack(pady=5)



# I launched the application.
root.mainloop()